<a id='sorted-sort'></a>

## 10. 🧩 Pattern 10: sorted() and .sort() with key= — LC 56, 57, 179, 252, 347, 435, 692, 973

---

```
PROBLEM:
  LC 56  — Merge Intervals: sort by start → intervals.sort(key=lambda x: x[0])
  LC 57  — Insert Interval: sort before merge pass
  LC 179 — Largest Number: custom comparator via cmp_to_key
  LC 252 — Meeting Rooms: sort by start, check consecutive overlap
  LC 253 — Meeting Rooms II: sort starts and ends separately
  LC 347 — Top K Frequent: sort by freq desc → sorted(d.items(), key=lambda x: -x[1])
  LC 435 — Non-Overlapping Intervals: sort by end time (greedy)
  LC 692 — Top K Frequent Words: sort by (-freq, word)
  LC 973 — K Closest Points: sort by distance²

sorted() vs .sort() — CRITICAL DIFFERENCE:
  sorted(iterable, key=..., reverse=...)   → returns NEW list, original unchanged
  list.sort(key=..., reverse=...)          → mutates IN PLACE, returns None

  ✅  result = sorted(nums)        ← safe, original preserved
  ❌  result = nums.sort()         ← result is None — common bug
  ✅  nums.sort()                  ← mutate in place when you don't need original

STABILITY — Python's Timsort is stable:
  Equal keys preserve original relative order.
  This makes multi-key sort via successive passes safe:
    first sort by secondary key, then sort by primary key → combined stable sort.
  Usually easier to use tuple key in a single pass instead.

key= ACCEPTS ANY CALLABLE:
  sorted(words, key=len)                     ← built-in function
  sorted(words, key=str.lower)               ← method reference
  sorted(words, key=lambda w: (-freq[w], w)) ← lambda
  sorted(items, key=operator.itemgetter(1))  ← itemgetter

MULTI-KEY TUPLE SORT — primary, secondary, tertiary:
  sorted(words, key=lambda s: (len(s), s))       ← by length, then alpha
  sorted(jobs,  key=lambda j: (j[1], j[0]))      ← by end, then start
  sorted(items, key=lambda x: (-x[1], x[0]))     ← by value desc, key asc

reverse=True vs NEGATION:
  sorted(nums, reverse=True)                 ← works for any comparable
  sorted(nums, key=lambda x: -x)             ← only works for numbers
  sorted(words, key=lambda w: (-len(w), w))  ← negate primary, keep secondary asc
  ⚠  can't do reverse=True on tuple keys with mixed direction — use negation

CUSTOM COMPARATOR — functools.cmp_to_key:
  from functools import cmp_to_key

  def compare(a, b):
      if str(a)+str(b) > str(b)+str(a): return -1   # a should come first
      if str(a)+str(b) < str(b)+str(a): return 1    # b should come first
      return 0

  sorted(nums, key=cmp_to_key(compare))   ← LC 179 pattern

SLOW MOTION TRACE — sort strings by length then alpha:
  words = ["fig", "apple", "ant", "cat"]
  key extractions:
    "fig"   → (3, "fig")
    "apple" → (5, "apple")
    "ant"   → (3, "ant")
    "cat"   → (3, "cat")
  tuple comparisons (primary length first):
    (3,"ant") < (3,"cat") < (3,"fig") < (5,"apple")
  result: ["ant", "cat", "fig", "apple"]

KEY INSIGHT:
  sorted() is non-destructive — prefer it when original order matters.
  .sort() is in-place — use it when you own the list and RAM matters.
  The key= function is called ONCE per element — not on every comparison.

TIME / SPACE:
  Time:  O(n log n) — Timsort
  Space: O(n) — sorted() allocates new list; .sort() is O(log n) stack space
```

In [ ]:
# Pattern 10: sorted() and .sort() with key=
# sorted() → new list; .sort() → in-place, returns None.

from functools import cmp_to_key
from collections import Counter

# 1. sorted() vs .sort() — the None trap
nums = [3, 1, 4, 1, 5, 9, 2, 6]
new_sorted  = sorted(nums)          # new list — nums unchanged
nums.sort()                         # in-place — nums is now sorted
bad_result  = [3, 1, 4].sort()     # ← returns None — common interview bug
print(f"sorted()    : {new_sorted}")
print(f"after .sort(): {nums}")
print(f"bad result  : {bad_result}")   # None

# 2. key= with built-in callables
words = ["fig", "apple", "ant", "cat", "banana"]
by_len    = sorted(words, key=len)            # built-in function as key
by_lower  = sorted(words, key=str.lower)      # method reference as key
print(f"by len    : {by_len}")
print(f"by lower  : {by_lower}")

# 3. multi-key tuple sort — by length then alpha
by_len_alpha = sorted(words, key=lambda s: (len(s), s))
# slow motion — key extractions:
# "fig"    → (3,"fig")
# "apple"  → (5,"apple")
# "ant"    → (3,"ant")
# "cat"    → (3,"cat")
# "banana" → (6,"banana")
# sorted tuples: (3,"ant")<(3,"cat")<(3,"fig")<(5,"apple")<(6,"banana")
print(f"len then alpha: {by_len_alpha}")

# 4. reverse=True vs negation — mixed direction needs negation
freq  = Counter(words)
# sort by freq DESC, then word ASC — can't use reverse=True (mixed directions)
by_freq_alpha = sorted(words, key=lambda w: (-freq[w], w))
print(f"freq desc then alpha: {by_freq_alpha}")

# 5. sort list of dicts by value desc, then key asc (drill)
scores = [{"name": "alice", "score": 90},
          {"name": "bob",   "score": 85},
          {"name": "carol", "score": 90},
          {"name": "dave",  "score": 85}]
ranked = sorted(scores, key=lambda d: (-d["score"], d["name"]))
print("ranked:")
for r in ranked:
    print(f"  {r['name']:6s} score={r['score']}")


def largest_number(nums: list) -> str:
    """
    LC 179 — Largest Number
    Approach: custom comparator — compare concatenated strings to decide order.
    Args:
        nums (list[int]): non-negative integers.
    Returns:
        str: largest number formed by arranging the integers.
    Time:  O(n log n * k) — sort with string comparison, k = avg digits
    Space: O(n) — string conversion
    """
    def compare(a, b):
        if a + b > b + a:   return -1   # a should come before b
        if a + b < b + a:   return 1    # b should come before a
        return 0

    # slow motion on [3, 30, 34, 5, 9]:
    # compare "9","5":  "95">"59" → 9 before 5
    # compare "5","34": "534">"345" → 5 before 34
    # compare "34","3": "343">"334" → 34 before 3
    # compare "3","30": "330">"303" → 3 before 30
    # sorted: ["9","5","34","3","30"] → "9534330"

    strs   = [str(n) for n in nums]
    strs.sort(key=cmp_to_key(compare))
    result = "".join(strs)
    return "0" if result[0] == "0" else result   # edge case: all zeros


def non_overlapping_intervals(intervals: list) -> int:
    """
    LC 435 — Non-Overlapping Intervals
    Approach: sort by end time (greedy) — earliest end leaves most room for rest.
    Args:
        intervals (list[list[int]]): list of [start, end] intervals.
    Returns:
        int: minimum number of intervals to remove.
    Time:  O(n log n) — dominated by sort
    Space: O(1) — in-place sort + counters
    """
    intervals.sort(key=lambda x: x[1])   # sort by END time — greedy key insight
    removed = 0
    prev_end = intervals[0][1]

    # slow motion on [[1,2],[2,3],[3,4],[1,3]]:
    # sorted by end: [[1,2],[2,3],[1,3],[3,4]]
    # prev_end=2
    # [2,3]: 2>=2 no overlap → prev_end=3
    # [1,3]: 1<3  OVERLAP   → remove it, removed=1 (keep earlier end)
    # [3,4]: 3>=3 no overlap → prev_end=4
    # result: 1

    for start, end in intervals[1:]:
        if start < prev_end:        # overlap — remove this interval
            removed += 1            # keep the one with earlier end (already prev_end)
        else:
            prev_end = end          # no overlap — advance the end boundary
    return removed


def test_harness(fn):
    tests = [
        ([[1, 2], [2, 3], [3, 4], [1, 3]], 1),   # remove [1,3]
        ([[1, 2], [1, 2], [1, 2]],          2),   # keep one, remove two
        ([[1, 2], [2, 3]],                  0),   # no overlap
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


test_harness(non_overlapping_intervals)

print(largest_number([3, 30, 34, 5, 9]))    # "9534330"
print(largest_number([10, 2]))              # "210"
print(largest_number([0, 0]))              # "0"

print("sorted_and_sort defined.")